# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client on one reporting day.

Time window: I will use March 2026 as the development and verification month. I will treat June 2026 as a sealed final test month and will not use the _sample table for label development.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))

HF_TOKEN loaded: True


In [2]:
!pip -q install duckdb huggingface_hub

In [7]:
import duckdb

con = duckdb.connect()

print("DuckDB connected:", con is not None)

DuckDB connected: True


In [8]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

print("Hugging Face login successful")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful


In [9]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=os.environ["HF_TOKEN"])

files = fs.glob("datasets/FlyRank/internship-warehouse/*")

print("Warehouse access successful.")
print("Files found:", len(files))
print(files[:5])

Warehouse access successful.
Files found: 7
['datasets/FlyRank/internship-warehouse/.gitattributes', 'datasets/FlyRank/internship-warehouse/README.md', 'datasets/FlyRank/internship-warehouse/dim_clients.parquet', 'datasets/FlyRank/internship-warehouse/dim_content.parquet', 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance']


In [10]:
files = fs.glob(
    "datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*"
)

print("Files found:", len(files))
print("\n".join(files[:10]))

Files found: 18
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-09
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-10


In [12]:
import os
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [os.environ["HF_TOKEN"]]
)

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [15]:
query = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(query).df()

print("Duplicate grain combinations found:", len(grain_check))
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,client_hash_id,content_hash_id,report_date,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions, clicks, sessions, users, engaged_sessions — historical performance signals available before the refresh decision.

Label / proxy: A future performance-decline outcome used to rank content items for refresh.

Context: content_id, client_id, report_date — used to identify content, group data, and define the time window, not as model features.

Excluded: trend_pct and trend_direction — excluded because they are used to derive the decline label and would leak the outcome into the features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain check: March 2026 has no duplicate client-content-date combinations, so the stated daily grain holds.

In [16]:
query = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(query).df()

print("Duplicate grain combinations found:", len(grain_check))
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,client_hash_id,content_hash_id,report_date,row_count


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Label/proxy: I will rank content items by their future performance decline/opportunity for refresh, using a future outcome as the label/proxy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.